# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



*   one row : one content per day for one client
*   table : fact_content_daily_peformance, dim_content
*   time window : 2025-01-01 - 2026-06-30





## 2. Fields: feature / label / context / excluded

*   feature : content_ages, days_since_update, impression_90_days, gsc_data_available, ga4_data_available_ clicks_90_days
*   label : future_decline

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [32]:
# Grain
dim_clients_schema = con.sql(f"""SELECT * FROM {TABLES['dim_clients']} LIMIT 1""").df().dtypes
fact_content_schema = con.sql(f"""SELECT * FROM {TABLES['fact_content_daily_performance']} LIMIT 1""").df().dtypes
date_content_client = con.sql(f"""SELECT COUNT(DISTINCT report_date) as report_date_total, COUNT(DISTINCT client_hash_id) as client_total, COUNT(DISTINCT content_hash_id) as content_total FROM {TABLES['fact_content_daily_performance']} """).df()


print("fact_content_daily_performance schema")
print(fact_content_schema,"\n")
print("dim_clients schema")
print(dim_clients_schema,"\n")
print(date_content_client.to_string(index=False),"\n")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance schema
report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                    bool
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     int64
gsc_avg_position                   float64
ga4_pageviews                        int64
ga4_sessions                         int64
ga4_users                            int64
ga4_engaged_sessions                 int64
ga4_total_engagement_sec             int64
sessions_organic                     int64
sessions_direct                      int64
sessions_referral                    int64
sessions_social                      int64
sessions_paid                        int64
sessions_ai                          int64
ai_chatgpt      

In [26]:
# Slice's row count and date span
count_row_date_span = con.sql(f"""SELECT
                                  COUNT(DISTINCT report_date) as report_date_total,
                                  COUNT(DISTINCT client_hash_id) as client_total,
                                  COUNT(DISTINCT content_hash_id) as content_total,
                                  MIN(report_date) as start_date,
                                  MAX(report_date) as end_date
                                  FROM
                                  {TABLES['fact_content_daily_performance']}
                                 """).df()


count_row_date_span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date_total,client_total,content_total,start_date,end_date
0,520,70,427292,2025-01-27,2026-06-30


In [33]:
# Availability
row_content_gsc_ga4 = con.sql(f"""SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT content_hash_id) AS total_content,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE) AS ga4_gsc_data_available,
            FROM {TABLES['fact_content_daily_performance']}""").df()

content_90_days_gsc_ga4 = con.sql(f"""
    SELECT
        COUNT(*) AS content_90_days_gsc_ga4
    FROM (
        SELECT
            content_hash_id
        FROM {TABLES['fact_content_daily_performance']}
        GROUP BY content_hash_id
        HAVING COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) >= 90
    ) AS content
""").df()

print(row_content_gsc_ga4.to_string(index=False))
print(content_90_days_gsc_ga4.to_string(index=False))
con.sql(f"""
    SELECT
        content_hash_id,
        MIN(report_date) as start_date_gsc_ga4,
        MAX(report_date) as end_date_gsc_ga4,
        COUNT(DISTINCT report_date) as range_gsc_ga4
    FROM (
        SELECT
            content_hash_id,
            report_date
        FROM {TABLES['fact_content_daily_performance']}
        WHERE
          gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
   )
   GROUP BY content_hash_id
   HAVING range_gsc_ga4 >= 90
   ORDER BY start_date_gsc_ga4, end_date_gsc_ga4
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  total_content  gsc_data_available  ga4_data_available  ga4_gsc_data_available
   78835655         427292            28970051             2816455                 2453375
 content_90_days_gsc_ga4
                    4619


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,start_date_gsc_ga4,end_date_gsc_ga4,range_gsc_ga4
0,content_6b4f9e61e903e486,2025-10-29,2026-04-28,99
1,content_d96a8c6c700e15c3,2025-10-29,2026-06-08,91
2,content_f3161b76ca328fd1,2025-10-29,2026-06-09,92
3,content_f0e5bb8c98cb95d7,2025-10-29,2026-06-10,114
4,content_1047d37c6f3935ba,2025-10-29,2026-06-12,161
...,...,...,...,...
4614,content_0e3f93e573464a71,2026-03-30,2026-06-30,92
4615,content_97ff27515379c0ca,2026-03-31,2026-06-29,91
4616,content_d1f8f969c7b729ec,2026-03-31,2026-06-30,92
4617,content_4d61b13ba2fff509,2026-03-31,2026-06-30,91


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.